# A Toy Atom-Interferometer Demo

This notebook connects the ideas developed throughout the repository to a simplified atom-interferometer model inspired by AION.

Previous notebooks showed:

- shared noise can cancel
- local noise remains
- correlation creates context
- baseline affects shared structure
- geometry determines measurable modes

Now we combine these ideas into a single differential sensing model.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
FIGURES = ROOT / "figures"
DATA = ROOT / "data"

FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

rng = np.random.default_rng(42)


## Two atom interferometers

Two interferometers share a common laser.

Each sensor sees:

- physical differential signal
- correlated laser phase noise
- local atom shot noise

The differential phase attempts to reject the common laser contribution.


In [ ]:
n = 1500
t = np.linspace(0, 10, n)

signal = 0.3 * np.sin(2 * np.pi * 0.4 * t)

laser_noise = 2.0 * rng.normal(size=n)

shot_a = 0.15 * rng.normal(size=n)
shot_b = 0.15 * rng.normal(size=n)

phi_a = signal / 2 + laser_noise + shot_a
phi_b = -signal / 2 + laser_noise + shot_b

phi_diff = phi_a - phi_b


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(t, phi_a, label="AI 1")
plt.plot(t, phi_b, label="AI 2", alpha=0.8)
plt.plot(t, phi_diff, label="Differential", linewidth=2)

plt.xlabel("time")
plt.ylabel("phase")

plt.title("Individual interferometers are noisy; differential readout recovers structure")

plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "37_individual_vs_differential.png",
    dpi=200
)

plt.show()


## Baseline and correlation

We model correlation decay as

ρ = exp(-baseline / coherence_length)

Longer baselines reduce the shared laser reference and therefore weaken common-mode rejection.


In [ ]:
baseline = np.linspace(0, 100, 80)
coherence_length = 50

rho = np.exp(-baseline / coherence_length)

shot_floor = []
total_error = []

for r in rho:

    shared = rng.normal(size=n)

    independent = rng.normal(size=n)

    laser_b = (
        r * shared +
        np.sqrt(1 - r**2) * independent
    )

    shot_a = 0.15 * rng.normal(size=n)
    shot_b = 0.15 * rng.normal(size=n)

    phi_a = signal/2 + shared + shot_a
    phi_b = -signal/2 + laser_b + shot_b

    phi_diff = phi_a - phi_b

    rmse = np.sqrt(
        np.mean((phi_diff - signal)**2)
    )

    total_error.append(rmse)

    shot_floor.append(
        np.sqrt(
            np.mean(
                (shot_a - shot_b)**2
            )
        )
    )


In [ ]:
plt.figure(figsize=(8,4))

plt.plot(
    baseline,
    total_error,
    label="Differential error"
)

plt.plot(
    baseline,
    shot_floor,
    linestyle="--",
    label="Shot-noise floor"
)

plt.xlabel("Baseline")
plt.ylabel("RMSE")

plt.title("Differential sensing approaches a shot-noise limit")

plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "37_baseline_sql_floor.png",
    dpi=200
)

plt.show()


## Noise budget

How much error remains under different conditions?


In [ ]:
laser_only = np.sqrt(
    np.mean(
        ((laser_noise - laser_noise) - 0)**2
    )
)

shot_only = np.sqrt(
    np.mean(
        ((shot_a - shot_b))**2
    )
)

combined = np.sqrt(
    np.mean(
        (phi_diff - signal)**2
    )
)

budget = pd.DataFrame({
    "component":[
        "laser only",
        "shot noise only",
        "differential readout"
    ],
    "rmse":[
        laser_only,
        shot_only,
        combined
    ]
})

budget


In [ ]:
plt.figure(figsize=(7,4))

plt.bar(
    budget["component"],
    budget["rmse"]
)

plt.ylabel("RMSE")

plt.title("Noise budget after differential sensing")

plt.tight_layout()

plt.savefig(
    FIGURES / "37_noise_budget.png",
    dpi=200
)

plt.show()


## AION-style summary

The shared laser produces large phase fluctuations.

Because the fluctuations are largely common to both interferometers, subtraction removes most of the disturbance.

What remains is primarily the differential signal plus local shot noise.


In [ ]:
summary = pd.DataFrame({
    "stage":[
        "shared laser",
        "interferometer A",
        "interferometer B",
        "differential readout",
        "remaining floor"
    ],
    "role":[
        "dominant common-mode disturbance",
        "phase measurement",
        "phase measurement",
        "common-mode rejection",
        "atom shot noise"
    ]
})

summary.to_csv(
    DATA / "37_aion_summary.csv",
    index=False
)

summary


# Lesson

The AION result is not that noise disappeared.

It is that the dominant injected laser phase noise had the right symmetry to be rejected.

Differential readout preserves the physical signal while leaving a shot-noise-limited floor.

Differential context makes precision possible.
